# Extraction Analysis

Recovery (recall/precision vs. ground truth) and hallucination rate for a single extraction run.

In [1]:
import sys
from pathlib import Path
%load_ext autoreload
%autoreload 2

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))
sys.path.insert(0, str(REPO_ROOT))

In [16]:
# ── Config ────────────────────────────────────────────────────────────────────
DATASET         = 'nfix'
MODEL           = 'gemma-3-27b'
EXTRACTION_DATE = '2026_04_20'  # set to None to use most recent run

In [17]:
import pandas as pd
from analysis.loaders import load_extraction, load_combined_judgements, load_ground_truth
from scholarlm.utils.unit_conversion import apply_unit_conversion
from analysis.metrics import recovery_rate, hallucination_rate, per_paper_metrics
from analysis.plots import recovery_bar, probability_distribution
from experiments.run_extraction import load_dataset_config

config = load_dataset_config(DATASET)

## Load data

In [18]:
records = load_extraction(DATASET, MODEL, EXTRACTION_DATE)
extraction_df = pd.DataFrame(records)
extraction_df = apply_unit_conversion(extraction_df, config.unit_conversion_table)
print(f'{len(extraction_df)} extracted measurements')
extraction_df.head()

5680 extracted measurements


,reference,doi,doi_data,url,publication_year,authors,title,publication,volume,pages,...,value,units,page_number,table_number,row_index,column_index,source,context,measurement_id,converted_value
0,Hicks and Silvester 1990 N Zealand J Mar Fresh...,10.1080/00288330.1990.9516439,NaN,http://www.tandfonline.com/doi/abs/10.1080/002...,1990,"Hicks, Brendan J.; Silvester, Warwick B.",Acetylene reduction associated with Zostera no...,New Zealand Journal of Marine and Freshwater R...,24,481-486,...,18,umol C₂H₄ m⁻² h⁻¹,[6],[None],[None],[None],[text],"[McClung, C. R.; Patriquin, D. G.; Davis, R. E...",0,18.000
1,Huang et al 2021 Catena 206,10.1016/j.catena.2021.105545,NaN,https://linkinghub.elsevier.com/retrieve/pii/S...,2021,"Huang, Fangjuan; Lin, Xianbiao; Hu, Weifang; Z...",Nitrogen cycling processes in sediments of the...,CATENA,206,105545,...,0.31,μg N g−¹ dry d−¹,[0],[None],[None],[None],[text],[Nitrogen cycling processes in sediments of th...,1,0.310
2,Huang et al 2021 Catena 206,10.1016/j.catena.2021.105545,NaN,https://linkinghub.elsevier.com/retrieve/pii/S...,2021,"Huang, Fangjuan; Lin, Xianbiao; Hu, Weifang; Z...",Nitrogen cycling processes in sediments of the...,CATENA,206,105545,...,4.43 × 10⁶,copies g⁻¹ sediment,[5],[None],[None],[None],[text],[Fig. 3. The spatial variations of sediment N ...,2,NaN
3,Huang et al 2021 Catena 206,10.1016/j.catena.2021.105545,NaN,https://linkinghub.elsevier.com/retrieve/pii/S...,2021,"Huang, Fangjuan; Lin, Xianbiao; Hu, Weifang; Z...",Nitrogen cycling processes in sediments of the...,CATENA,206,105545,...,2.025,μg N g⁻¹,[7],[None],[None],[None],[text],[Fig. 6. Sediment physicochemical properties (...,3,2.025
4,Huang et al 2021 Catena 206,10.1016/j.catena.2021.105545,NaN,https://linkinghub.elsevier.com/retrieve/pii/S...,2021,"Huang, Fangjuan; Lin, Xianbiao; Hu, Weifang; Z...",Nitrogen cycling processes in sediments of the...,CATENA,206,105545,...,0.31,μg N g−1 dry d−1,[11],[None],[None],[None],[text],[large gap is still remaining in determining i...,4,0.310


In [19]:
extraction_df.attribute

0            nfix_rate_areal
1             nfix_rate_mass
2             nfix_rate_mass
3             nfix_rate_mass
4             nfix_rate_mass
                ...         
5675    nfix_rate_volumetric
5676    nfix_rate_volumetric
5677    nfix_rate_volumetric
5678    nfix_rate_volumetric
5679    nfix_rate_volumetric
Name: attribute, Length: 5680, dtype: str

In [5]:
ground_truth_df = load_ground_truth(config)
print(f'{len(ground_truth_df)} ground truth measurements')
ground_truth_df.head()

3410 ground truth measurements


,author,title,name,location,ecosystem,date,state,attribute,value
0,chopyk et al.,agricultural freshwater pond supports diverse ...,NaN,central maryland; usa,ponds,NaN,NaN,max_depth,3.35
1,chopyk et al.,agricultural freshwater pond supports diverse ...,NaN,central maryland; usa,ponds,NaN,NaN,ph,7.78
2,chopyk et al.,agricultural freshwater pond supports diverse ...,NaN,central maryland; usa,ponds,NaN,NaN,surface_area,2600.00
3,tudor; m.; tudor; i-m; ibram; o.; teodorof; l....,analysis of biological indicators related to t...,cuibul cu lebede lake,danube delta,shallow lake,NaN,NaN,ph,8.03
4,tudor; m.; tudor; i-m; ibram; o.; teodorof; l....,analysis of biological indicators related to t...,isac lake,danube delta,shallow lake,NaN,NaN,ph,7.75


In [ ]:
judgements = load_combined_judgements(DATASET, MODEL, EXTRACTION_DATE)
judged_df = pd.DataFrame(judgements)
print(f'{len(judged_df)} judged measurements')

## Recovery

In [12]:
# Update strict_matching to match your dataset's entity key columns
if 'pond' in DATASET:
    STRICT_MATCHING = {'title': 'title', 'attribute': 'attribute', 'converted_value': 'value'}
    FUZZY_MATCHING  = {'name': 'name', 'location': 'location', 'ecosystem': 'ecosystem'}
elif 'nfix' in DATASET:
    extraction_df['attribute'] = extraction_df['attribute'].map({'nfix_rate_areal': 'nfix_rate', 'nfix_rate_volumetric': 'nfix_rate', 'nfix_rate_mass': 'nfix_rate'})
    STRICT_MATCHING = {'paper_code': 'reference_id', 'attribute': 'attribute', 'converted_value': 'value'}
    FUZZY_MATCHING  = {"name": "site_name", "site_type": "habitat"}

stats = recovery_rate(
    extraction_df,
    ground_truth_df,
    strict_matching=STRICT_MATCHING,
    fuzzy_matching=FUZZY_MATCHING,
    cache_path=Path(f"../data/experiments/{DATASET}/extraction/{MODEL}/{EXTRACTION_DATE}/match_cache.pkl")
)
print(stats)

{'recovery': 0.4026392961876833, 'precision': 0.061503314818132954, 'n_extracted': 22324, 'n_gt': 3410}


## Hallucination rate

In [ ]:
hall = hallucination_rate(judged_df)
print(hall)

fig = probability_distribution(judged_df, prob_col='judgement_p_true', label_col='judgement_combined')
fig.savefig('figures/prob_distribution.pdf', bbox_inches='tight')
fig.show()

## Per-paper breakdown

In [ ]:
paper_df = per_paper_metrics(
    extraction_df,
    ground_truth_df,
    judged_df=judged_df,
    strict_matching=STRICT_MATCHING,
    fuzzy_matching=FUZZY_MATCHING,
)
paper_df.sort_values('recall').round(3)